In [ ]:
# 1. Install the library

from sentence_transformers import SentenceTransformer, util

# 2. Load a lightweight, high-performance model
# 'all-MiniLM-L6-v2' is a great balance of speed and accuracy for Colab
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Use case 1: Semantic search (query -> most relevant sentence)
# Each sentence is a possible answer in a small knowledge base.
knowledge_base = [
    "Reset your password from Settings > Security.",
    "To cancel a subscription, open Billing and click Cancel plan.",
    "You can export reports as CSV from the Analytics page.",
    "Enable two-factor authentication for extra account protection."
]

# User question phrased differently from the matching knowledge-base sentence (tests semantic match).
query = "How do I stop my monthly plan?"

# Encode documents and query into dense vectors in the same embedding space.
kb_embeddings = model.encode(knowledge_base, convert_to_tensor=True)
query_embedding = model.encode(query, convert_to_tensor=True)

# Cosine similarity between query and each KB sentence; [0] is the vector of scores for this query.
scores = util.cos_sim(query_embedding, kb_embeddings)[0]
best_idx = int(scores.argmax())

print("Query:", query)
print("Best match:", knowledge_base[best_idx])
print(f"Score: {scores[best_idx].item():.4f}")

In [ ]:
# Use case 2: Intent routing (map user text to closest intent label)
# Each intent label has one example sentence that represents that intent.
intent_examples = {
    "billing_issue": "I was charged too much this month",
    "password_reset": "I cannot log in and need to reset my password",
    "cancel_subscription": "Please cancel my account immediately",
    "feature_request": "Can you add dark mode support?"
}

# This is the new user message we want to classify.
incoming_text = "I want to stop paying and close my plan"

# Separate the intent labels from their example texts.
labels = list(intent_examples.keys())
prototypes = list(intent_examples.values())

# Convert the intent examples and incoming message into semantic embedding vectors.
prototype_embeddings = model.encode(prototypes, convert_to_tensor=True)
incoming_embedding = model.encode(incoming_text, convert_to_tensor=True)

# Compare the incoming message to each intent example using cosine similarity.
intent_scores = util.cos_sim(incoming_embedding, prototype_embeddings)[0]

# Pick the intent whose example text is most semantically similar to the incoming message.
best_intent_idx = int(intent_scores.argmax())

# Display the original text, predicted intent label, and similarity score.
print("Incoming text:", incoming_text)
print("Predicted intent:", labels[best_intent_idx])
print(f"Confidence score: {intent_scores[best_intent_idx].item():.4f}")

In [ ]:
# Use case 3: Duplicate ticket detection (find near-duplicate messages)
# This is the new support ticket we want to compare against older tickets.
new_ticket = "My invoice is much higher than expected"

# These are existing tickets that may or may not describe the same issue.
existing_tickets = [
    "I think my bill is too high this month",
    "The app crashes when I open reports",
    "Please help me change my account email",
    "Why was I charged more than usual?"
]

# Convert both the existing tickets and the new ticket into semantic embedding vectors.
existing_embeddings = model.encode(existing_tickets, convert_to_tensor=True)
new_ticket_embedding = model.encode(new_ticket, convert_to_tensor=True)

# Compare the new ticket to every existing ticket using cosine similarity.
dupe_scores = util.cos_sim(new_ticket_embedding, existing_embeddings)[0]

# Sort tickets from highest to lowest similarity score before printing.
ranked_tickets = sorted(
    zip(existing_tickets, dupe_scores),
    key=lambda item: item[1].item(),
    reverse=True
)

# Print each existing ticket with its similarity score to inspect possible duplicates.
print("New ticket:", new_ticket)
for ticket, score in ranked_tickets:
    print(f"- {ticket} -> {score.item():.4f}")

# The highest-scoring existing ticket is the most likely duplicate.
likely_dupe_idx = int(dupe_scores.argmax())
print("\nMost likely duplicate:", existing_tickets[likely_dupe_idx])